## **Agentic Vision-Language Safety Reasoning for Explainable Detection of Non-PPE Violations in Dynamic Construction Sites**

### **Streaming Dataset Load**

In [ ]:
from datasets import load_dataset

# Stream to avoid pulling the full 4.5GB dataset, then materialize the first 5000 examples
raw_stream = load_dataset("LouisChen15/ConstructionSite", split="train", streaming=True)

subset = []
for i, example in enumerate(raw_stream):
    if i >= 5000:
        break
    subset.append(example)
    if (i + 1) % 500 == 0:
        print(f"...loaded {i + 1} images so far")

print(f"Loaded {len(subset)} images")
print(subset[0].keys())

README.md:   0%|          | 0.00/7.33k [00:00<?, ?B/s]

...loaded 500 images so far
...loaded 1000 images so far
...loaded 1500 images so far
...loaded 2000 images so far
...loaded 2500 images so far
...loaded 3000 images so far
...loaded 3500 images so far
...loaded 4000 images so far
...loaded 4500 images so far
...loaded 5000 images so far
Loaded 5000 images
dict_keys(['image', 'image_id', 'image_caption', 'illumination', 'camera_distance', 'view', 'quality_of_info', 'rule_1_violation', 'rule_2_violation', 'rule_3_violation', 'rule_4_violation', 'excavator', 'rebar', 'worker_with_white_hard_hat'])


### **Label Extraction & DataFrame Construction**

In [ ]:
import numpy as np
import pandas as pd

def extract_rule_label(example, rule_key):
    """Rule violation field is either None/Null or a dict with bounding_box + reason."""
    v = example.get(rule_key)
    return 1 if v is not None else 0

def extract_bbox(example, rule_key):
    v = example.get(rule_key)
    if v is None:
        return []
    return v.get("bounding_box", [])

def extract_reason(example, rule_key):
    v = example.get(rule_key)
    if v is None:
        return ""
    return v.get("reason", "")

records = []
for ex in subset:
    record = {
        "image": ex["image"],                      # PIL.Image
        "image_id": ex["image_id"],
        "image_caption": ex["image_caption"],
        "illumination": ex["illumination"],
        "camera_distance": ex["camera_distance"],
        "view": ex["view"],
        "quality_of_info": ex["quality_of_info"],
        "rule_1": extract_rule_label(ex, "rule_1_violation"),
        "rule_2": extract_rule_label(ex, "rule_2_violation"),
        "rule_3": extract_rule_label(ex, "rule_3_violation"),
        "rule_4": extract_rule_label(ex, "rule_4_violation"),
        "bbox_rule_1": extract_bbox(ex, "rule_1_violation"),
        "bbox_rule_2": extract_bbox(ex, "rule_2_violation"),
        "bbox_rule_3": extract_bbox(ex, "rule_3_violation"),
        "bbox_rule_4": extract_bbox(ex, "rule_4_violation"),
        "reason_rule_1": extract_reason(ex, "rule_1_violation"),
        "reason_rule_2": extract_reason(ex, "rule_2_violation"),
        "reason_rule_3": extract_reason(ex, "rule_3_violation"),
        "reason_rule_4": extract_reason(ex, "rule_4_violation"),
        "excavator": ex.get("excavator", []),
        "rebar": ex.get("rebar", []),
        "worker_with_white_hard_hat": ex.get("worker_with_white_hard_hat", []),
    }
    records.append(record)

df = pd.DataFrame(records)

# Multi-label target matrix [Rule1, Rule2, Rule3, Rule4]
label_cols = ["rule_1", "rule_2", "rule_3", "rule_4"]
Y = df[label_cols].values.astype(int)

# Overall violation flag (any rule violated)
df["is_violation"] = (Y.sum(axis=1) > 0).astype(int)

print(df[label_cols + ["is_violation"]].sum())
print(df.shape)

rule_1          476
rule_2           48
rule_3           91
rule_4           36
is_violation    631
dtype: int64
(5000, 23)


### **Multilabel Stratified Train/Val/Test Split**

In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

X_idx = np.arange(len(df)).reshape(-1, 1)

# Step 1: split off test (15%)
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
trainval_idx, test_idx = next(msss1.split(X_idx, Y))

# Step 2: split remaining into train (70% overall) / val (15% overall)
Y_trainval = Y[trainval_idx]
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=42)
train_rel_idx, val_rel_idx = next(msss2.split(np.zeros(len(trainval_idx)).reshape(-1,1), Y_trainval))
def save_split(df_split, split_name):
    meta = []
    for i, row in df_split.iterrows():
        img_path = f"data/images/{row['image_id']}.jpg"
        row["image"].convert("RGB").save(img_path)
        meta.append({
            "image_id": row["image_id"],
            "image_path": img_path,
            "image_caption": row["image_caption"],
            "illumination": row["illumination"],
            "camera_distance": row["camera_distance"],
            "view": row["view"],
            "quality_of_info": row["quality_of_info"],
            "labels": [int(row[c]) for c in label_cols],
            "bboxes": {c: row[f"bbox_{c}"] for c in label_cols},
            "reasons": {c: row[f"reason_{c}"] for c in label_cols},
        })
train_idx = trainval_idx[train_rel_idx]
val_idx   = trainval_idx[val_rel_idx]

df_train = df.iloc[train_idx].reset_index(drop=True)
df_val   = df.iloc[val_idx].reset_index(drop=True)
df_test  = df.iloc[test_idx].reset_index(drop=True)

print("Train:", len(df_train), "Val:", len(df_val), "Test:", len(df_test))
print("Train rule distribution:\n", df_train[label_cols].sum())
print("Val rule distribution:\n", df_val[label_cols].sum())
print("Test rule distribution:\n", df_test[label_cols].sum())

### **Persisting Images & Metadata to Disk**

In [ ]:
import os, json

os.makedirs("data/images", exist_ok=True)
with open(f"data/{split_name}.json", "w") as f:
        json.dump(meta, f, indent=2)
    return meta

train_meta = save_split(df_train, "train")
val_meta   = save_split(df_val, "val")
test_meta  = save_split(df_test, "test")

### **Safety Knowledge Base**

In [ ]:
SAFETY_KNOWLEDGE_BASE = {
    "rule_1": {
        "context": "worker on foot at construction site",
        "required_entities": ["worker"],
        "spatial_relationships": ["worker wearing/not-wearing PPE item"],
        "required_evidence": ["hard hat", "covered shoulders/legs", "closed-toe shoes", "hi-vis vest at night", "eye/face protection when cutting/welding/grinding/drilling"],
        "violation_condition": "worker on foot missing one or more required PPE items"
    },
    "rule_2": {
        "context": "work at height >= 3 meters without edge protection",
        "required_entities": ["worker", "height/edge"],
        "spatial_relationships": ["worker near unprotected edge at height"],
        "required_evidence": ["safety harness"],
        "violation_condition": "worker at height >=3m near unprotected edge without harness"
    },
    "rule_3": {
        "context": "excavation or elevated edge requiring protection",
        "required_entities": ["edge", "excavation/retaining wall"],
        "spatial_relationships": ["edge adjacent to walkable/standable area"],
        "required_evidence": ["guardrail", "fence", "edge warning"],
        "violation_condition": "excavation >=3m depth or steep retaining wall edge without guardrail/fence/warning"
    },
    class ConstructionSiteDataset(Dataset):
    def __init__(self, meta_list, transform=None):
        self.meta = meta_list
        self.transform = transform

    "rule_4": {
        "context": "excavator in operation",
        "required_entities": ["excavator", "worker"],
        "spatial_relationships": ["worker within blind spot / operation radius of excavator"],
        "required_evidence": ["excavator operating", "worker proximity"],
        "violation_condition": "worker present in blind spot or operation radius of an active excavator"
    },
}

### **PyTorch Dataset Class for Scene Attributes**

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image

ILLUM_CLASSES = sorted(df["illumination"].unique().tolist())
DIST_CLASSES  = sorted(df["camera_distance"].unique().tolist())
VIEW_CLASSES  = sorted(df["view"].unique().tolist())
QUAL_CLASSES  = sorted(df["quality_of_info"].unique().tolist())

def to_idx(val, classes):
    return classes.index(val)

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        item = self.meta[idx]
        image = Image.open(item["image_path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        target = {
            "labels": torch.tensor(item["labels"], dtype=torch.float32),
            "illumination": to_idx(item["illumination"], ILLUM_CLASSES),
            "camera_distance": to_idx(item["camera_distance"], DIST_CLASSES),
            "view": to_idx(item["view"], VIEW_CLASSES),
            "quality_of_info": to_idx(item["quality_of_info"], QUAL_CLASSES),
            "caption": item["image_caption"],
            "image_id": item["image_id"],
            "bboxes": item["bboxes"],
            "reasons": item["reasons"],
        }
        return image, target
def forward(self, pixel_values):
        # Ensure pixel_values are on the correct device and have the expected float type.
        # This is a common source of TypeErrors in deep learning when inputs are not perfectly aligned.
        pixel_values = pixel_values.to(self.clip.device, dtype=torch.float32)

        with torch.no_grad():
            # The clip.get_image_features method usually returns a tensor.
            # However, the error indicates it's returning a BaseModelOutputWithPooling.
            # We need to extract the actual feature tensor from it.
            clip_output = self.clip.get_image_features(pixel_values=pixel_values)
            feats = clip_output.pooler_output if hasattr(clip_output, 'pooler_output') else clip_output

### **Agent 1-Scene Understanding Agent (CLIP-based) + Training Loop**

In [ ]:
import torch.nn as nn
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

class SceneUnderstandingAgent(nn.Module):
    def __init__(self, clip_model, embed_dim=512):
        super().__init__()
        self.clip = clip_model
        for p in self.clip.parameters():
            p.requires_grad = False  # freeze CLIP backbone

        self.illum_head = nn.Linear(embed_dim, len(ILLUM_CLASSES))
        self.dist_head  = nn.Linear(embed_dim, len(DIST_CLASSES))
        self.view_head  = nn.Linear(embed_dim, len(VIEW_CLASSES))
        self.qual_head  = nn.Linear(embed_dim, len(QUAL_CLASSES))
        return {
            "illumination": self.illum_head(feats),
            "camera_distance": self.dist_head(feats),
            "view": self.view_head(feats),
            "quality_of_info": self.qual_head(feats),
            "embedding": feats,
        }

device = "cuda" if torch.cuda.is_available() else "cpu"
scene_agent = SceneUnderstandingAgent(clip_model).to(device)

def collate_scene(batch):
    images, targets = zip(*batch)
    inputs = clip_processor(images=list(images), return_tensors="pt")
    return inputs["pixel_values"], targets

from torch.utils.data import DataLoader
train_ds = ConstructionSiteDataset(train_meta)
val_ds   = ConstructionSiteDataset(val_meta)
test_ds  = ConstructionSiteDataset(test_meta)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_scene)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, collate_fn=collate_scene)

optimizer = torch.optim.Adam(
    list(scene_agent.illum_head.parameters()) +
    list(scene_agent.dist_head.parameters()) +
    list(scene_agent.view_head.parameters()) +
    list(scene_agent.qual_head.parameters()), lr=1e-4
)
ce_loss = nn.CrossEntropyLoss()

EPOCHS = 5
for epoch in range(EPOCHS):
    scene_agent.train()
    total_loss = 0
    for pixel_values, targets in train_loader:
        pixel_values = pixel_values.to(device)

        illum_labels = torch.tensor([t["illumination"] for t in targets]).to(device)
        dist_labels  = torch.tensor([t["camera_distance"] for t in targets]).to(device)
        view_labels  = torch.tensor([t["view"] for t in targets]).to(device)
        qual_labels  = torch.tensor([t["quality_of_info"] for t in targets]).to(device)

        out = scene_agent(pixel_values)
        loss = (ce_loss(out["illumination"], illum_labels) +
                ce_loss(out["camera_distance"], dist_labels) +
                ce_loss(out["view"], view_labels) +
                ce_loss(out["quality_of_info"], qual_labels))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"[Scene Agent] Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/len(train_loader):.4f}")

### **Checkpointing the Scene Agent**

In [ ]:
# SAVE scene_agent
os.makedirs("checkpoints", exist_ok=True)
torch.save(scene_agent.state_dict(), "checkpoints/scene_agent.pt")

# Save the class label lists too — you need these exact lists (in this exact order)
# to decode predictions later, and they're only known after looking at df.
with open("checkpoints/label_classes.json", "w") as f:
    for img_file in image_files:
    img = Image.open(img_file)

    json.dump({
        "ILLUM_CLASSES": ILLUM_CLASSES,
        "DIST_CLASSES": DIST_CLASSES,
        "VIEW_CLASSES": VIEW_CLASSES,
        "QUAL_CLASSES": QUAL_CLASSES,
    }, f, indent=2)

print("Saved scene_agent + label classes.")

### **Scene Agent Performance Plot**

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

image_files = ["agent1_scene_accuracy.png"]
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis("off")
    # plt.title(img_file)   # Removed so no filename is displayed
    plt.show()

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

image_files = ["/content/agent1_scene_loss.png"]
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
for img_file in image_files:
    img = Image.open(img_file)

    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis("off")
    # plt.title(img_file)   # Removed so no filename is displayed
    plt.show()

### **Evaluation Metrics - Agent 1**

In [ ]:
import pandas as pd

# Display all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# Load Excel file (reads ALL sheets into a dictionary of DataFrames)
# Display each sheet found in the file
for sheet_name, df in excel_file.items():
    print(f"SHEET: {sheet_name}")
    print(df)
    print("\n")
    print("Shape :", df.shape)
    print("Columns :", list(df.columns))


###  **Agent 2-Contextual Reasoning**

In [ ]:
def contextual_reasoning_agent(scene_output, detected_entities, knowledge_base=SAFETY_KNOWLEDGE_BASE):
    """
    Symbolic reasoning: decides which rules are *applicable* given scene context,
    before the detection agent decides which are *violated*.
    scene_output: dict with 'view', 'camera_distance', etc. (decoded strings)
    detected_entities: dict e.g. {'excavator': [...], 'worker': [...], 'edge': bool}
    """
    applicable_rules = []

    if detected_entities.get("worker_count", 0) > 0:
        applicable_rules.append("rule_1")

    if detected_entities.get("worker_count", 0) > 0 and scene_output.get("view") in ["elevation view"]:
        applicable_rules.append("rule_2")

    if detected_entities.get("edge_present", False):
        applicable_rules.append("rule_3")

    if detected_entities.get("excavator_count", 0) > 0 and detected_entities.get("worker_count", 0) > 0:
        applicable_rules.append("rule_4")

    evidence = {r: knowledge_base[r]["required_evidence"] for r in applicable_rules}
    return {"applicable_rules": applicable_rules, "supporting_evidence": evidence}


### **Focal Loss Definition & Class-imbalance Handling**

In [ ]:
# CELL 9
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha  # tensor shape [4]: weight given to the POSITIVE class per rule
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p_t = torch.exp(-bce)
        focal_term = (1 - p_t) ** self.gamma

        if self.alpha is not None:
            # alpha applied per-sample based on whether the true label is 1 or 0
            alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
            focal_term = focal_term * alpha_t

        pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
# alpha = fraction of negatives → higher alpha means "positive class is rarer, weight it more"
alpha = neg_counts / total
alpha = torch.clamp(torch.tensor(alpha, dtype=torch.float32), 0.1, 0.9).to(device)

focal_loss_fn = FocalLoss(alpha=alpha, gamma=2.0)
RULE_THRESHOLDS = {"rule_1": 0.5, "rule_2": 0.35, "rule_3": 0.35, "rule_4": 0.35}  # will be re-tuned after retraining

### **Agent 3-Detector, Rule Classifier & T5 Explanation Model**

In [ ]:
import os
import torchvision
import torchvision.transforms.functional as TF
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs("checkpoints", exist_ok=True)

NUM_CLASSES = 5  # background + 4 rules

detector = fasterrcnn_resnet50_fpn(weights="DEFAULT")
in_features = detector.roi_heads.box_predictor.cls_score.in_features
detector.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
detector = detector.to(device)

# Multi-label rule classifier head, fed by CLIP embeddings from the scene agent
class RuleClassifier(nn.Module):
    def __init__(self, embed_dim=512, num_rules=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_rules)
        )
    def forward(self, embedding):
        return self.net(embedding)

rule_classifier = RuleClassifier().to(device)

# T5 explanation generator
t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")
t5_model = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)

def scale_bbox_to_pixels(norm_box, width, height):
    x1, y1, x2, y2 = norm_box
    return [x1 * width, y1 * height, x2 * width, y2 * height]


# Detection dataset: returns full-resolution image tensors + boxes
# scaled to that same image's pixel space (torchvision handles the
# internal resize itself and adjusts boxes to match).

class DetectionDataset(Dataset):
    def __init__(self, meta_list):
        self.meta = meta_list

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        item = self.meta[idx]
        image = Image.open(item["image_path"]).convert("RGB")
        width, height = image.size
        image_tensor = TF.to_tensor(image)  # C x H x W, values in [0,1]

        boxes, labels = [], []
        for i, rule in enumerate(label_cols, start=1):
            for bb in item["bboxes"][rule]:
                px_box = scale_bbox_to_pixels(bb, width, height)
                if px_box[2] > px_box[0] and px_box[3] > px_box[1]:  # guard against degenerate boxes
                    boxes.append(px_box)
                    labels.append(i)

        if len(boxes) == 0:
            boxes_t = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes_t = torch.tensor(boxes, dtype=torch.float32)
            labels_t = torch.tensor(labels, dtype=torch.int64)

        return image_tensor, {"boxes": boxes_t, "labels": labels_t}

def detection_collate(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)

detection_train_ds = DetectionDataset(train_meta)
detection_train_loader = DataLoader(detection_train_ds, batch_size=4, shuffle=True, collate_fn=detection_collate)
# Detector training loop
detector_optimizer = torch.optim.SGD(detector.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
DETECTOR_EPOCHS = 10

detector.train()
for epoch in range(DETECTOR_EPOCHS):
    total_loss = 0
    n_batches = 0
    for images, targets in detection_train_loader:
        if all(t["boxes"].shape[0] == 0 for t in targets):
            continue  # nothing to localize in this batch, skip

        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = detector(images, targets)
        loss = sum(loss_dict.values())

        detector_optimizer.zero_grad()
        loss.backward()
        detector_optimizer.step()
        total_loss += loss.item()
        n_batches += 1

    avg_loss = total_loss / max(n_batches, 1)
    print(f"[Detector] Epoch {epoch+1}/{DETECTOR_EPOCHS} - Loss: {avg_loss:.4f} (batches with boxes: {n_batches})")

torch.save(detector.state_dict(), "checkpoints/detector.pt")


# Rule-classifier training (unchanged logic, uses fixed focal_loss_fn from Cell 9)
rule_optimizer = torch.optim.Adam(rule_classifier.parameters(), lr=1e-4)

for epoch in range(EPOCHS):
    rule_classifier.train()
    total_loss = 0
    for pixel_values, targets in train_loader:
        pixel_values = pixel_values.to(device)
        labels = torch.stack([t["labels"] for t in targets]).to(device)

        with torch.no_grad():
            clip_output = scene_agent.clip.get_image_features(pixel_values=pixel_values)
            embedding = clip_output.pooler_output if hasattr(clip_output, 'pooler_output') else clip_output

        logits = rule_classifier(embedding)
        loss = focal_loss_fn(logits, labels)

        rule_optimizer.zero_grad()
        loss.backward()
        rule_optimizer.step()
        total_loss += loss.item()

    print(f"[Rule Classifier] Epoch {epoch+1}/{EPOCHS} - Focal Loss: {total_loss/len(train_loader):.4f}")

torch.save(rule_classifier.state_dict(), "checkpoints/rule_classifier.pt")


# T5 explanation fine-tuning (unchanged)
for img_file in image_files:
    img = Image.open(img_file)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
        t5_optimizer.zero_grad()
        loss.backward()
        t5_optimizer.step()
        total_loss += loss.item()

    print(f"[T5 Explanation] Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/max(len(batches),1):.4f}")

torch.save(t5_model.state_dict(), "checkpoints/t5_model.pt")

### **Rule Classifier Performance Plot**

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis("off")
    # plt.title(img_file)   # Removed so no filename is displayed
    plt.show()

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
for img_file in image_files:
    img = Image.open(img_file)

    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis("off")
    # plt.title(img_file)   # Removed so no filename is displayed
    plt.show()
    from PIL import Image
import matplotlib.pyplot as plt

image_files = ["/content/agent3a_detector_map.png"]

for img_file in image_files:
    img = Image.open(img_file)

    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis("off")
    # plt.title(img_file)   # Removed so no filename is displayed
    plt.show()

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
for img_file in image_files:
    img = Image.open(img_file)

    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis("off")
    # plt.title(img_file)   # Removed so no filename is displayed
    plt.show()

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
for img_file in image_files:
    img = Image.open(img_file)

    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis("off")
    # plt.title(img_file)   # Removed so no filename is displayed
    plt.show()

### **Evaluation Metrics** **- Agent 3**

In [ ]:
import pandas as pd

# Display all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Load Excel file (reads ALL sheets into a dictionary of DataFrames)
excel_file = pd.read_excel("/content/Agent 3-Detection and Explanation.xlsx", sheet_name=None)
def make_verification_sample(item, embedding, correct=True):
    rule_idx = None
    for i, rule in enumerate(label_cols):
        if item["labels"][i] == 1:
            rule_idx = i
            break
    if rule_idx is None:
        return None

    rule_onehot = torch.zeros(4)
    bbox_list = item["bboxes"][label_cols[rule_idx]]
    bbox = bbox_list[0] if bbox_list else [0, 0, 0, 0]

# Display each sheet found in the file
for sheet_name, df in excel_file.items():
    print(f"SHEET: {sheet_name}")
    print(df)
    print("\n")
    print("Shape :", df.shape)
    print("Columns :", list(df.columns))


### **Verification Agent**

In [ ]:
import random

class VerificationAgent(nn.Module):
    def __init__(self, in_dim=512 + 4 + 4 + 1, hidden=128, num_classes=3):
        # input: image_embedding + rule_onehot(4) + bbox_features(4: x1,y1,x2,y2 normalized) + text_sim(1)
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, num_classes)  # Supported=0, Partially=1, Unsupported=2
        )
    def forward(self, x):
        return self.net(x)

verifier = VerificationAgent().to(device)
verifier_optimizer = torch.optim.Adam(verifier.parameters(), lr=1e-4)
verifier_loss_fn = nn.CrossEntropyLoss()



    if correct:
        rule_onehot[rule_idx] = 1
        label = 0  # Supported
    else:
        # controlled incorrect combination: wrong rule id and/or shuffled bbox
        wrong_idx = random.choice([i for i in range(4) if i != rule_idx])
        rule_onehot[wrong_idx] = 1
        bbox = [random.random() for _ in range(4)]
        label = random.choice([1, 2])  # Partially or Unsupported

    text_sim = torch.tensor([1.0 if correct else random.uniform(0.0, 0.5)])
    feat = torch.cat([embedding, rule_onehot, torch.tensor(bbox, dtype=torch.float32), text_sim])
    return feat, label

for epoch in range(EPOCHS):
    verifier.train()
    total_loss = 0
    n_batches = 0
    for pixel_values, targets in train_loader:
        pixel_values = pixel_values.to(device)
        with torch.no_grad():
            clip_output = scene_agent.clip.get_image_features(pixel_values=pixel_values)
            embeddings = (clip_output.pooler_output if hasattr(clip_output, 'pooler_output') else clip_output).cpu()

        feats, labels_v = [], []
        for emb, item in zip(embeddings, targets):
            pos = make_verification_sample(item, emb, correct=True)
            neg = make_verification_sample(item, emb, correct=False)
            for sample in [pos, neg]:
                if sample is not None:
                    feats.append(sample[0])
                    labels_v.append(sample[1])

        if not feats:
            continue

        feats = torch.stack(feats).to(device)
        labels_v = torch.tensor(labels_v).to(device)

        logits = verifier(feats)
        loss = verifier_loss_fn(logits, labels_v)

        verifier_optimizer.zero_grad()
        loss.backward()
        verifier_optimizer.step()
        total_loss += loss.item()
        n_batches += 1

    print(f"[Verifier] Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/max(n_batches,1):.4f}")

### **Checkpointing the Verifier**

In [ ]:
#  SAVE verifier
torch.save(verifier.state_dict(), "checkpoints/verifier.pt")
print("Saved verifier.")

### **Prediction**

In [ ]:
import torch, json
from PIL import Image

def run_pipeline_verbose(image_path, caption_hint="", detected_entities=None):
    if detected_entities is None:
        # In a full system this would come from an object-detection pass;
        # here it's a placeholder you can wire up to `detector`'s output boxes.
        detected_entities = {"worker_count": 1, "excavator_count": 0, "edge_present": False}

    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=[image], return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)

    with torch.no_grad():
        # AGENT 1: Scene Understanding
        scene_out = scene_agent(pixel_values)
        embedding = scene_out["embedding"]
        scene_info = {
            "illumination": ILLUM_CLASSES[scene_out["illumination"].argmax().item()],
            "camera_distance": DIST_CLASSES[scene_out["camera_distance"].argmax().item()],
            "view": VIEW_CLASSES[scene_out["view"].argmax().item()],
            "quality_of_info": QUAL_CLASSES[scene_out["quality_of_info"].argmax().item()],
        }

        #  AGENT 2: Contextual Reasoning
        reasoning_out = contextual_reasoning_agent(scene_info, detected_entities=detected_entities)

        # AGENT 3: Detection + Rule Classification + Explanation
        rule_logits = torch.sigmoid(rule_classifier(embedding)).cpu().numpy()[0]
        confidence_scores = {label_cols[i]: float(rule_logits[i]) for i in range(4)}
        predicted_rules = [label_cols[i] for i in range(4) if rule_logits[i] >= RULE_THRESHOLDS[label_cols[i]]]

        detector.eval()
        det_preds = detector([TF.to_tensor(image).to(device)])[0]
        # keep only reasonably confident boxes
        keep = det_preds["scores"] > 0.5
        detected_boxes = det_preds["boxes"][keep].cpu().tolist()
        detected_labels = [label_cols[l - 1] for l in det_preds["labels"][keep].cpu().tolist()]

        explanations = {}
        for rule in predicted_rules:
            prompt = f"rule: {rule} caption: {caption_hint}"
            enc = t5_tokenizer(prompt, return_tensors="pt").to(device)
            gen = t5_model.generate(**enc, max_length=40)
            explanations[rule] = t5_tokenizer.decode(gen[0], skip_special_tokens=True)

        # AGENT 4: Verification
        rule_onehot = torch.zeros(4)
        bbox_feat = torch.zeros(4)
        if predicted_rules:
            idx = label_cols.index(predicted_rules[0])
            rule_onehot[idx] = 1
            if detected_boxes:
                x1, y1, x2, y2 = detected_boxes[0]
                w, h = image.size
                bbox_feat = torch.tensor([x1 / w, y1 / h, x2 / w, y2 / h])
        verify_feat = torch.cat([embedding.cpu()[0], rule_onehot, bbox_feat, torch.tensor([1.0])])
        verify_logits = verifier(verify_feat.unsqueeze(0).to(device))
        verify_label = ["Supported", "Partially Supported", "Unsupported"][verify_logits.argmax().item()]

    return {
        "agent_1_scene_understanding": scene_info,
        "agent_2_contextual_reasoning": reasoning_out,
        "agent_3_detection_and_explanation": {
            "confidence_scores": confidence_scores,
            "predicted_violations": predicted_rules if predicted_rules else "No violation crossed threshold",
            "detected_boxes": list(zip(detected_labels, [ [round(c,1) for c in b] for b in detected_boxes ])),
            "explanations": explanations if explanations else "N/A (no violation predicted)",
        },
        "agent_4_verification": verify_label,
    }


def print_agent_report(result):
    print("=" * 60)
    print(f"Agent 1 (Scene Understanding): {result['agent_1_scene_understanding']}")
    print("-" * 60)
    print(f"Agent 2 (Contextual Reasoning): {result['agent_2_contextual_reasoning']}")
    print("-" * 60)
    print("Agent 3 (Detection + Rule Classifier + Explanation):")
    for k, v in result["agent_3_detection_and_explanation"].items():
        print(f"   {k}: {v}")
    print("-" * 60)
    print(f"Agent 4 (Verification): {result['agent_4_verification']}")
    print("=" * 60)
# Read the CSV
# Print each row completely
for i, row in df.iterrows():
    print(f" {i+1}")
    for col in df.columns:
        print(f"{col}: {row[col]}")



